# A8: Semantic relationships and retrieval (TakeOff destination guides)

A retrieval-augmented generation (RAG) system for **TakeOff**, our trip-planning app, tested with three embedding
models and three chunking strategies.

**Pipeline:** Data → Chunks → Embeddings → Vector store → Top-k retrieval → Prompt → SmolLM2-1.7B → Answer

| Component | Choice |
|---|---|
| Knowledge base | `data/destination_guides.md`: guides for the 12 cities in TakeOff's destination list (40 paragraphs) |
| Extra data (scaling test only) | `data/extra_guides.md`: TakeOff help pages and general travel advice (19 paragraphs) |
| Embedding models | `all-MiniLM-L6-v2` (384 dims), `all-mpnet-base-v2` (768), `BAAI/bge-large-en-v1.5` (1024) |
| Generation model | `HuggingFaceTB/SmolLM2-1.7B-Instruct`, from our A6/A7 benchmark, the same for every experiment |
| Chunking | fixed length, overlapping paragraphs, hybrid (section-aware with a context header) |

Run with **Restart & Run All**. The first run downloads about 5 GB of model weights from Hugging Face. Every
result table is also written to `results/` so it can be read outside the notebook.

In [1]:
import os, re, json, time, math, platform, random
from pathlib import Path
from collections import defaultdict

# Progress bars from model downloads open many sockets inside Jupyter and can hit macOS's open-file limit
# ("Too many open files"), so they are switched off before any Hugging Face library is imported.
os.environ.setdefault('HF_HUB_DISABLE_PROGRESS_BARS', '1')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

import numpy as np
import pandas as pd

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.width', 220)

DATA_DIR = Path('data')
RESULTS_DIR = Path('results')
RESULTS_DIR.mkdir(exist_ok=True)

# Set RAG_DRY_RUN=1 to test the notebook logic with tiny stand-in models (no downloads). Real runs leave it unset.
DRY_RUN = os.environ.get('RAG_DRY_RUN') == '1'

EMBEDDING_MODELS = {   # size label -> Hugging Face id
    'small':  'sentence-transformers/all-MiniLM-L6-v2',
    'medium': 'sentence-transformers/all-mpnet-base-v2',
    'large':  'BAAI/bge-large-en-v1.5',
}
# BGE models are trained to see this instruction in front of short search queries (from the model card).
QUERY_PREFIX = {'large': 'Represent this sentence for searching relevant passages: '}

GEN_MODEL = 'HuggingFaceTB/SmolLM2-1.7B-Instruct'
TOP_K = 3              # chunks put into the prompt in the main experiment
MAX_NEW_TOKENS = 120   # answers are 2-3 sentences
SEED = 42
random.seed(SEED); np.random.seed(SEED)

if DRY_RUN:
    DEVICE = 'cpu'
    print('DRY RUN: stand-in models, results are NOT real')
else:
    import torch
    torch.manual_seed(SEED)
    DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
    import transformers, sentence_transformers
    print('torch', torch.__version__, '| transformers', transformers.__version__,
          '| sentence-transformers', sentence_transformers.__version__)
print('device', DEVICE, '|', platform.platform(), '| python', platform.python_version())

W1109 16:38:11.535000 17301 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


torch 2.15.0 | transformers 5.22.2 | sentence-transformers 6.2.0
device mps | macOS-26.6.2-arm64-arm-64bit-Mach-O | python 3.14.7


## Part 1: Build the RAG pipeline

### Step 1.1: Knowledge base

TakeOff members plan trips through twelve cities (Chicago, New York, Rome, Florence, Venice, Milan, Tokyo, Kyoto,
Osaka, Lisbon, Porto, Reykjavik). The questions they ask while planning are about those cities: how to get from the
airport, what to eat, when to go, what to book ahead. So the knowledge base is one guide document per city, written
for this assignment from general travel knowledge, with the average daily cost taken from TakeOff's own
destination table.

**Format.** `## City, Country` starts a document, `### Topic` starts a section, and a **paragraph** is a block of
text separated by a blank line. Every paragraph gets an id such as `kyoto-1` (city slug plus position in that city).

In [2]:
def slug(text):
    return re.sub(r'[^a-z]+', '-', text.lower()).strip('-')

def load_paragraphs(path):
    # Parse a guide file into paragraph records with city, topic and a unique id.
    paragraphs, city, topic, counter = [], None, None, defaultdict(int)
    for block in re.split(r'\n\s*\n', Path(path).read_text(encoding='utf-8')):
        block = ' '.join(block.split())
        if block.startswith('## '):
            city = block[3:]; continue
        if block.startswith('### '):
            topic = block[4:]; continue
        if not block or block.startswith('# ') or city is None:
            continue
        key = slug(city.split(',')[0])
        counter[key] += 1
        paragraphs.append(dict(pid=f'{key}-{counter[key]}', doc=city, topic=topic, text=block,
                               words=len(block.split()), source=Path(path).name))
    return paragraphs

PARAGRAPHS = load_paragraphs(DATA_DIR / 'destination_guides.md')
EXTRA = load_paragraphs(DATA_DIR / 'extra_guides.md')
PARA_BY_ID = {p['pid']: p for p in PARAGRAPHS + EXTRA}

kb = pd.DataFrame(PARAGRAPHS)
print(f"Documents: {kb.doc.nunique()} | paragraphs: {len(kb)} | words: {kb.words.sum()} | "
      f"paragraph length {kb.words.min()}-{kb.words.max()} words (mean {kb.words.mean():.0f})")
assert len(kb) >= 20 and 2000 <= kb.words.sum() <= 5000 and kb.words.between(80, 200).all()
print('Size requirements met: >= 20 paragraphs, 80-200 words each, 2,000-5,000 words in total')
display(kb.groupby('doc', sort=False).agg(paragraphs=('pid', 'count'), words=('words', 'sum'),
                                          topics=('topic', lambda t: ', '.join(t))))

Documents: 12 | paragraphs: 40 | words: 3732 | paragraph length 80-121 words (mean 93)
Size requirements met: >= 20 paragraphs, 80-200 words each, 2,000-5,000 words in total


,paragraphs,words,topics
doc,,,
"Chicago, United States",3,321,"Getting around, Sights and things to do, Food, seasons and costs"
"New York, United States",4,382,"Getting around, Sights and things to do, Shows and evenings, Food, seasons and costs"
"Rome, Italy",4,393,"Getting around, Ancient Rome and the Vatican, Food, seasons and costs, Practical tips"
"Florence, Italy",3,264,"Getting around, Sights and museums, Food, day trips and costs"
"Venice, Italy",3,295,"Getting around, Sights and islands, Food, seasons and costs"
"Milan, Italy",3,274,"Getting around, Sights and day trips, Food, evenings and costs"
"Tokyo, Japan",4,364,"Getting around, Neighborhoods and sights, Food, Seasons, money and costs"
"Kyoto, Japan",3,271,"Getting there and around, Temples, shrines and districts, Food, seasons and costs"
"Osaka, Japan",3,269,"Getting around, Food and nightlife, Sights and day trips"


### Step 1.2: Text representation and chunking

All three strategies share two rules from the brief:

* **No sentence is split.** Text is first cut into sentences (a regex on `.`, `!` or `?` followed by a space and a
  capital letter, with abbreviations such as `St.` protected), and chunks are built only from whole sentences.
* **No unrelated topics are merged.** A chunk never crosses from one city's document into another's.

| Strategy | How it works |
|---|---|
| **1. Fixed length** | Each city document is read as one stream of sentences. Sentences are added to the current chunk until the next one would take it past **100 words**, then a new chunk starts. Chunk size is fixed; topic boundaries inside a city are ignored. |
| **2. Overlapping paragraphs** | A sliding window of **two paragraphs with a stride of one**: chunk 1 = paragraph 1 + 2, chunk 2 = paragraph 2 + 3, and so on inside each city. Every paragraph except the first and last of a city appears in two chunks, so the overlap is one full paragraph (about 50% of each chunk). |
| **3. Hybrid (section-aware)** | One chunk per paragraph, because each paragraph is one topic. Each chunk gets a **context header** with its city and topic (`Kyoto, Japan / Getting there and around:`), so a paragraph that never names its city still carries it. Paragraphs over 150 words would be split at the sentence boundary nearest the middle (none of ours are that long). |

In [3]:
_ABBREV = re.compile(r'\b(St|Mt|Dr|Mr|Mrs|Ms|vs|approx|e\.g|i\.e)\.', re.I)

def sentences(text):
    protected = _ABBREV.sub(lambda m: m.group(0).replace('.', '<DOT>'), text)
    parts = re.split(r'(?<=[.!?])\s+(?=[A-Z0-9"])', protected)
    return [p.replace('<DOT>', '.').strip() for p in parts if p.strip()]

def by_doc(paragraphs):
    groups = defaultdict(list)
    for p in paragraphs:
        groups[p['doc']].append(p)
    return groups

def chunk_fixed(paragraphs, max_words=100):
    chunks = []
    for doc, paras in by_doc(paragraphs).items():
        current, pids = [], []
        for p in paras:
            for s in sentences(p['text']):
                n = len(s.split())
                if current and sum(len(c.split()) for c in current) + n > max_words:
                    chunks.append(dict(text=' '.join(current), doc=doc, pids=sorted(set(pids))))
                    current, pids = [], []
                current.append(s); pids.append(p['pid'])
        if current:
            chunks.append(dict(text=' '.join(current), doc=doc, pids=sorted(set(pids))))
    return chunks

def chunk_overlap(paragraphs, window=2, stride=1):
    chunks = []
    for doc, paras in by_doc(paragraphs).items():
        if len(paras) <= window:
            starts = [0]
        else:
            starts = range(0, len(paras) - window + 1, stride)
        for i in starts:
            group = paras[i:i + window]
            chunks.append(dict(text=' '.join(p['text'] for p in group), doc=doc, pids=[p['pid'] for p in group]))
    return chunks

def chunk_hybrid(paragraphs, split_over=150):
    chunks = []
    for p in paragraphs:
        header = f"{p['doc']} / {p['topic']}: "
        sents = sentences(p['text'])
        if p['words'] > split_over and len(sents) > 1:   # split near the middle, on a sentence boundary
            half, run, cut = p['words'] / 2, 0, 1
            for i, s in enumerate(sents):
                run += len(s.split())
                if run >= half:
                    cut = max(1, i + 1); break
            pieces = [' '.join(sents[:cut]), ' '.join(sents[cut:])]
        else:
            pieces = [p['text']]
        for piece in pieces:
            chunks.append(dict(text=header + piece, doc=p['doc'], pids=[p['pid']]))
    return chunks

CHUNKERS = {'fixed': chunk_fixed, 'overlap': chunk_overlap, 'hybrid': chunk_hybrid}

def build_chunks(paragraphs):
    out = {}
    for name, fn in CHUNKERS.items():
        out[name] = [dict(c, cid=f'{name}-{i}', words=len(c['text'].split())) for i, c in enumerate(fn(paragraphs))]
    return out

CHUNKS = build_chunks(PARAGRAPHS)

def check_chunks(chunks, paragraphs):
    # Every chunk must be made of whole sentences from one document.
    all_sentences = {s for p in paragraphs for s in sentences(p['text'])}
    for c in chunks:
        body = c['text'].split(': ', 1)[1] if c['cid'].startswith('hybrid') else c['text']
        assert all(s in all_sentences for s in sentences(body)), c['cid']
        assert len({PARA_BY_ID[pid]['doc'] for pid in c['pids']}) == 1, c['cid']
    return True

rows = []
for name, chunks in CHUNKS.items():
    check_chunks(chunks, PARAGRAPHS)
    w = [c['words'] for c in chunks]
    rows.append(dict(strategy=name, chunks=len(chunks), mean_words=round(np.mean(w)), min_words=min(w),
                     max_words=max(w), total_words=sum(w),
                     paragraphs_split_across_chunks=sum(1 for p in PARAGRAPHS
                         if sum(p['pid'] in c['pids'] for c in chunks) > 1 and name == 'fixed')))
chunk_stats = pd.DataFrame(rows)
display(chunk_stats)
print('Checks passed: every chunk is whole sentences from a single city document.')
for name in CHUNKS:
    c = CHUNKS[name][3]
    print(f"\n[{name}] {c['cid']} covers {c['pids']} ({c['words']} words)\n{c['text'][:400]}...")

,strategy,chunks,mean_words,min_words,max_words,total_words,paragraphs_split_across_chunks
0,fixed,44,85,10,100,3732,20
1,overlap,28,185,163,226,5168,0
2,hybrid,40,100,87,127,3995,0


Checks passed: every chunk is whole sentences from a single city document.

[fixed] fixed-3 covers ['chicago-3'] (74 words)
Summer is the busiest season, with free concerts in Millennium Park and festivals almost every weekend; Lollapalooza fills Grant Park in early August and hotel prices jump that week. Winters are long, cold and windy, so visitors from December to March should pack a warm coat, hat and gloves. In TakeOff's destination data a typical day in Chicago costs about 170 US dollars per person, including a m...

[overlap] overlap-3 covers ['new-york-2', 'new-york-3'] (176 words)
Central Park covers 843 acres and is best explored on foot or by rented bike. The Metropolitan Museum of Art sits on its eastern edge, and the American Museum of Natural History is on the western side. The High Line, a park built on an old elevated railway, runs through the Meatpacking District and Chelsea. Ferries to the Statue of Liberty and Ellis Island leave from Battery Park, and tickets that...

`total_words` shows the price of overlap: the overlapping strategy stores most paragraphs twice, so it embeds and
stores about 1.4 times as much text as the other two (5,168 words against 3,732). In the fixed strategy, `paragraphs_split_across_chunks`
counts paragraphs whose sentences ended up in two different chunks; those are places where a topic is cut in half.

### Step 1.3: Embedding pipeline

Each chunk is turned into a vector with each of the three models and stored in a small vector store: a matrix of
L2-normalised vectors plus the chunk records. With normalised vectors, cosine similarity is a single matrix product,
which is exact and fast at this size (a library such as FAISS would replace it at millions of chunks).

**Size categories** (our own, by output dimension and parameter count): small = 384 dimensions and about 23M
parameters, medium = 768 and about 110M, large = 1024 and about 335M.

In [4]:
class HashEmbedder:
    # Stand-in used only when DRY_RUN is set: bag of words hashed into a fixed-size vector.
    def __init__(self, dim):
        self.dim = dim
    def get_sentence_embedding_dimension(self):
        return self.dim
    def encode(self, texts, normalize_embeddings=True, **_):
        out = np.zeros((len(texts), self.dim), dtype=np.float32)
        for i, t in enumerate(texts):
            for w in re.findall(r'[a-z0-9]+', t.lower()):
                out[i, hash(w) % self.dim] += 1
        return out / np.maximum(np.linalg.norm(out, axis=1, keepdims=True), 1e-9)

def load_embedder(model_id):
    if DRY_RUN:
        return HashEmbedder({'MiniLM': 384, 'mpnet': 768}.get(next((k for k in ('MiniLM', 'mpnet') if k in model_id), ''), 1024))
    from sentence_transformers import SentenceTransformer
    return SentenceTransformer(model_id, device=DEVICE)

def count_params(model):
    return sum(p.numel() for p in model.parameters()) if hasattr(model, 'parameters') else 0

class VectorStore:
    # L2-normalised embedding matrix + chunk records; cosine search by dot product.
    def __init__(self, embedder, size, chunks):
        self.embedder, self.size, self.chunks = embedder, size, chunks
        t0 = time.perf_counter()
        self.matrix = np.asarray(embedder.encode([c['text'] for c in chunks], normalize_embeddings=True,
                                                 batch_size=16, show_progress_bar=False), dtype=np.float32)
        self.build_seconds = time.perf_counter() - t0

    def embed_query(self, query):
        q = QUERY_PREFIX.get(self.size, '') + query
        return np.asarray(self.embedder.encode([q], normalize_embeddings=True, show_progress_bar=False),
                          dtype=np.float32)[0]

    def search(self, query, k=TOP_K, docs=None):
        # docs: optional set of 'City, Country' names; only chunks from those documents are scored
        t0 = time.perf_counter()
        allowed = np.array([i for i, c in enumerate(self.chunks) if docs is None or c['doc'] in docs])
        scores = self.matrix[allowed] @ self.embed_query(query)
        top = np.argsort(-scores)[:k]
        ms = (time.perf_counter() - t0) * 1000
        return [dict(self.chunks[allowed[i]], score=float(scores[i]), rank=r + 1) for r, i in enumerate(top)], ms

EMBEDDERS, STORES, emb_rows = {}, {}, []
for size, model_id in EMBEDDING_MODELS.items():
    t0 = time.perf_counter()
    EMBEDDERS[size] = load_embedder(model_id)
    load_s = time.perf_counter() - t0
    for strategy, chunks in CHUNKS.items():
        store = VectorStore(EMBEDDERS[size], size, chunks)
        STORES[(size, strategy)] = store
        emb_rows.append(dict(size=size, model=model_id.split('/')[-1],
                             dims=EMBEDDERS[size].get_sentence_embedding_dimension(),
                             params_m=round(count_params(EMBEDDERS[size]) / 1e6, 1), strategy=strategy,
                             chunks=len(chunks), build_s=round(store.build_seconds, 2),
                             ms_per_chunk=round(1000 * store.build_seconds / len(chunks), 1),
                             index_kb=round(store.matrix.nbytes / 1024, 1), load_s=round(load_s, 1)))
embedding_table = pd.DataFrame(emb_rows)
embedding_table.to_csv(RESULTS_DIR / 'embedding_table.csv', index=False)
display(embedding_table)

/var/folders/gx/jnc6dq8s4j7_m1xg50dc170c0000gn/T/ipykernel_17301/1334251600.py:55: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  dims=EMBEDDERS[size].get_sentence_embedding_dimension(),
/var/folders/gx/jnc6dq8s4j7_m1xg50dc170c0000gn/T/ipykernel_17301/1334251600.py:55: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  dims=EMBEDDERS[size].get_sentence_embedding_dimension(),
/var/folders/gx/jnc6dq8s4j7_m1xg50dc170c0000gn/T/ipykernel_17301/1334251600.py:55: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  dims=EMBEDDERS[size].get_sentence_embedding_dimension(),


,size,model,dims,params_m,strategy,chunks,build_s,ms_per_chunk,index_kb,load_s
0,small,all-MiniLM-L6-v2,384,22.7,fixed,44,0.23,5.3,66.0,0.5
1,small,all-MiniLM-L6-v2,384,22.7,overlap,28,0.11,3.9,42.0,0.5
2,small,all-MiniLM-L6-v2,384,22.7,hybrid,40,0.09,2.3,60.0,0.5
3,medium,all-mpnet-base-v2,768,109.5,fixed,44,0.59,13.4,132.0,0.8
4,medium,all-mpnet-base-v2,768,109.5,overlap,28,0.83,29.5,84.0,0.8
5,medium,all-mpnet-base-v2,768,109.5,hybrid,40,0.60,14.9,120.0,0.8
6,large,bge-large-en-v1.5,1024,335.1,fixed,44,1.72,39.1,176.0,1.9
7,large,bge-large-en-v1.5,1024,335.1,overlap,28,2.26,80.6,112.0,1.9
8,large,bge-large-en-v1.5,1024,335.1,hybrid,40,1.81,45.4,160.0,1.9


### Step 1.4: Retrieval

`VectorStore.search` embeds the query with the same model (adding the BGE query instruction for the large model),
scores every chunk by cosine similarity and returns the top-k with their scores. Example with the medium model and
hybrid chunks:

In [5]:
example_hits, example_ms = STORES[('medium', 'hybrid')].search('How do I get from the airport to Kyoto?', k=3)
print(f'search time {example_ms:.1f} ms')
for h in example_hits:
    print(f"#{h['rank']} {h['cid']} {h['pids']} score={h['score']:.3f}: {h['text'][:110]}...")

search time 28.8 ms
#1 hybrid-24 ['kyoto-1'] score=0.797: Kyoto, Japan / Getting there and around: Kyoto has no airport of its own. International travelers usually fly ...
#2 hybrid-27 ['osaka-1'] score=0.649: Osaka, Japan / Getting around: Osaka is served by Kansai International Airport (KIX), built on an island in Os...
#3 hybrid-25 ['kyoto-2'] score=0.572: Kyoto, Japan / Temples, shrines and districts: Fushimi Inari Taisha, famous for thousands of red torii gates c...


### Step 1.5: Generation

The prompt has three parts, as the brief asks: a **system instruction**, the **retrieved context** (each chunk
labelled with its source) and the **user query**. The generator is SmolLM2-1.7B-Instruct, the model TakeOff
already runs for day-by-day drafts. In our A6 benchmark it had the best itinerary score (0.758) and the fastest
time per itinerary (3.2 s) of the 15 models, so it is the realistic choice for a feature that will live inside the
same Django server. Decoding is greedy (no sampling) so every configuration is compared on identical generation
settings.

In [6]:
SYSTEM_PROMPT = ("You are TakeOff's travel assistant. Answer the traveler's question using only the context below. "
                 "If the context does not contain the answer, say that you do not know. "
                 "Answer in two or three sentences.")

def build_messages(query, hits, system=SYSTEM_PROMPT, best_last=False):
    # best_last puts the highest-scoring passage directly before the question (small models weigh recent text most)
    ordered = list(reversed(hits)) if best_last else hits
    context = '\n\n'.join(f"[{i + 1}] {h['text']}" for i, h in enumerate(ordered))
    return [dict(role='system', content=system),
            dict(role='user', content=f"Context:\n{context}\n\nQuestion: {query}")]

class EchoGenerator:
    # Stand-in for DRY_RUN: answers with the first sentence of the first context chunk.
    def count_tokens(self, messages):
        return int(sum(len(m['content'].split()) for m in messages) * 1.3)
    def __call__(self, messages, max_new_tokens=MAX_NEW_TOKENS):
        ctx = messages[-1]['content'].split('\n\n')[0]
        return sentences(ctx.split('] ', 1)[-1])[0]

class LocalGenerator:
    def __init__(self, model_id):
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer
        self.torch = torch
        dtype = torch.float16 if DEVICE != 'cpu' else torch.bfloat16
        self.tok = AutoTokenizer.from_pretrained(model_id)
        self.model = AutoModelForCausalLM.from_pretrained(model_id, dtype=dtype).to(DEVICE).eval()
    def _encode(self, messages):
        return self.tok.apply_chat_template(messages, add_generation_prompt=True, return_tensors='pt',
                                            return_dict=True).to(DEVICE)
    def count_tokens(self, messages):
        return int(self._encode(messages)['input_ids'].shape[1])
    def __call__(self, messages, max_new_tokens=MAX_NEW_TOKENS):
        inputs = self._encode(messages)
        with self.torch.no_grad():
            out = self.model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                                      repetition_penalty=1.1,
                                      pad_token_id=self.tok.pad_token_id or self.tok.eos_token_id)
        return self.tok.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

t0 = time.perf_counter()
GENERATOR = EchoGenerator() if DRY_RUN else LocalGenerator(GEN_MODEL)
print(f'generator ready in {time.perf_counter() - t0:.1f} s')

def answer(query, store, k=TOP_K, system=SYSTEM_PROMPT, rewrite=None, docs=None, best_last=False):
    search_query = rewrite(query) if rewrite else query
    hits, retrieval_ms = store.search(search_query, k=k, docs=docs)
    messages = build_messages(query, hits, system, best_last)
    t0 = time.perf_counter()
    reply = GENERATOR(messages)
    return dict(hits=hits, answer=reply, retrieval_ms=retrieval_ms, generation_s=time.perf_counter() - t0,
                prompt_tokens=GENERATOR.count_tokens(messages))

demo = answer('How do I get from the airport to Kyoto?', STORES[('medium', 'hybrid')])
print(demo['answer'])
print(f"retrieval {demo['retrieval_ms']:.0f} ms | generation {demo['generation_s']:.1f} s | prompt {demo['prompt_tokens']} tokens")

generator ready in 6.9 s
To get from Kansai International Airport (KIX) to Kyoto, you can take the JR Haruka limited express train, which takes about 75 to 80 minutes to reach Kyoto Station. Alternatively, you can also rent a car at the airport and drive to Kyoto, but this may be more time-consuming due to traffic. Another option is to use taxis or ride-hailing services, but these options tend to be more expensive than public transportation.
retrieval 139 ms | generation 4.8 s | prompt 502 tokens


## Part 2: System experiments

### Step 2.1: Test queries

Eight questions a TakeOff member could ask while planning. Each has the paragraph(s) that hold the answer
(`gold`) and the facts a correct answer must contain (`facts`; each fact is a group of acceptable words). Some are
deliberately hard:

* **Q1** asks for "the airport" in Kyoto, which has none; the answer is in a paragraph about Kansai airport.
* **Q5** needs a "no": July has the midnight sun, so no northern lights.
* **Q7** needs facts from two cities (Lisbon and Porto), so one retrieved chunk is not enough.
* **Q8** never names the place that answers it (Nara, in the Osaka guide).

In [7]:
QUERIES = [
    dict(qid='Q1', query='How do I get from the airport to Kyoto?', gold=['kyoto-1'],
         facts=[['kansai', 'kix'], ['haruka']]),
    dict(qid='Q2', query='What local food should I try in Lisbon?', gold=['lisbon-3'],
         facts=[['nata'], ['bacalhau', 'cod', 'sardine', 'bifana']]),
    dict(qid='Q3', query='Will Venice be flooded if I visit in November?', gold=['venice-3'],
         facts=[['acqua alta', 'high tide', 'high water'], ['mose', 'barrier']]),
    dict(qid='Q4', query='Do I need to book the Last Supper in Milan ahead of time?', gold=['milan-2'],
         facts=[['advance', 'ahead', 'early', 'book'], ['weeks', 'sell out', 'sold out']]),
    dict(qid='Q5', query='Can I see the northern lights on a July trip to Reykjavik?', gold=['reykjavik-3'],
         facts=[['no', 'not', "can't", 'cannot', 'unlikely'], ['dark', 'midnight sun', 'september', 'april']]),
    dict(qid='Q6', query='What is a cheap way to ride the subway a lot in New York?', gold=['new-york-1'],
         facts=[['omny', 'cap', 'capped'], ['12', 'twelve', 'free']]),
    dict(qid='Q7', query='Which is cheaper per day, Lisbon or Porto?', gold=['lisbon-3', 'porto-3'],
         facts=[['porto'], ['85'], ['95']]),
    dict(qid='Q8', query='Where can I see free-roaming deer on a day trip from Osaka?', gold=['osaka-3'],
         facts=[['nara'], ['todai', 'buddha', 'park']]),
]
display(pd.DataFrame(QUERIES)[['qid', 'query', 'gold']])

,qid,query,gold
0,Q1,How do I get from the airport to Kyoto?,[kyoto-1]
1,Q2,What local food should I try in Lisbon?,[lisbon-3]
2,Q3,Will Venice be flooded if I visit in November?,[venice-3]
3,Q4,Do I need to book the Last Supper in Milan ahead of time?,[milan-2]
4,Q5,Can I see the northern lights on a July trip to Reykjavik?,[reykjavik-3]
5,Q6,What is a cheap way to ride the subway a lot in New York?,[new-york-1]
6,Q7,"Which is cheaper per day, Lisbon or Porto?","[lisbon-3, porto-3]"
7,Q8,Where can I see free-roaming deer on a day trip from Osaka?,[osaka-3]


### Step 2.2: Evaluation table (3 embedding models × 3 chunking strategies × 8 queries)

How each column is measured, the same way for every row:

| Column | Measure |
|---|---|
| Retrieved | the top-3 chunk ids with the paragraphs they cover and their similarity score |
| Gold hit / rank | whether any retrieved chunk contains a gold paragraph, and the rank of the first one |
| Recall | share of the query's gold paragraphs found in the top 3 |
| **Retrieved context quality (1-10)** | 1 + 9 × share of the required facts present in the retrieved text |
| **Answer quality (1-10)** | 1 + 9 × share of the required facts present in the answer |
| Unsupported numbers | numbers in the answer that appear nowhere in the retrieved context (a hallucination signal) |
| Latency | query embedding + search (ms) and generation (s) |

The scores are computed by a script so all 72 rows are judged the same way; we read the answers as well, and the
failure analysis is based on reading them.

In [8]:
def has_fact(text, group):
    t = ' ' + re.sub(r'[^a-z0-9 ]+', ' ', text.lower()) + ' '
    return any((' ' + re.sub(r'[^a-z0-9 ]+', ' ', g.lower()) + ' ') in t if len(g) <= 3 else g.lower() in text.lower()
               for g in group)

def fact_share(text, facts):
    return sum(has_fact(text, g) for g in facts) / len(facts)

def numbers(text):
    return set(re.findall(r'\d+(?:[.,]\d+)?', text))

def evaluate(q, result, size, strategy, k=TOP_K, label='main'):
    hits, ans = result['hits'], result['answer']
    context = ' '.join(h['text'] for h in hits)
    covered = {pid for h in hits for pid in h['pids']}
    gold_ranks = [h['rank'] for h in hits if set(h['pids']) & set(q['gold'])]
    unsupported = sorted(numbers(ans) - numbers(context) - numbers(q['query']))
    ctx_share, ans_share = fact_share(context, q['facts']), fact_share(ans, q['facts'])
    notes = []
    if not gold_ranks: notes.append('gold paragraph not retrieved')
    elif ans_share < ctx_share: notes.append('answer dropped facts that were in the context')
    if unsupported: notes.append('unsupported numbers ' + ', '.join(unsupported))
    if re.search(r"(do not|don't) know|not (mentioned|provided|available)", ans.lower()): notes.append('said it does not know')
    return dict(run=label, embedding=size, chunking=strategy, k=k, qid=q['qid'], query=q['query'],
                retrieved=' | '.join(f"{h['cid']}{h['pids']} {h['score']:.2f}" for h in hits),
                gold_hit=bool(gold_ranks), gold_rank=min(gold_ranks) if gold_ranks else None,
                recall=round(len(covered & set(q['gold'])) / len(q['gold']), 2),
                context_quality=round(1 + 9 * ctx_share, 1), answer_quality=round(1 + 9 * ans_share, 1),
                unsupported_numbers=', '.join(unsupported), retrieval_ms=round(result['retrieval_ms'], 1),
                generation_s=round(result['generation_s'], 2), prompt_tokens=result['prompt_tokens'],
                answer=ans, notes='; '.join(notes),
                retrieved_text=' ||| '.join(f"[{h['cid']}] {h['text']}" for h in hits))

rows, t_start = [], time.perf_counter()
for (size, strategy), store in STORES.items():
    for q in QUERIES:
        rows.append(evaluate(q, answer(q['query'], store), size, strategy))
    print(f'{size:>6} / {strategy:<7} done ({time.perf_counter() - t_start:.0f} s elapsed)')
results = pd.DataFrame(rows)
results.to_csv(RESULTS_DIR / 'evaluation_table.csv', index=False)

 small / fixed   done (24 s elapsed)
 small / overlap done (56 s elapsed)
 small / hybrid  done (76 s elapsed)
medium / fixed   done (99 s elapsed)
medium / overlap done (131 s elapsed)
medium / hybrid  done (152 s elapsed)
 large / fixed   done (175 s elapsed)
 large / overlap done (205 s elapsed)
 large / hybrid  done (224 s elapsed)


In [9]:
# The evaluation table (full retrieved chunk text for every row is in results/evaluation_table.csv)
cols = ['embedding', 'chunking', 'qid', 'retrieved', 'gold_rank', 'context_quality', 'answer_quality',
        'retrieval_ms', 'generation_s', 'answer', 'notes']
with pd.option_context('display.max_rows', 100, 'display.max_colwidth', 90):
    display(results[cols])

,embedding,chunking,qid,retrieved,gold_rank,context_quality,answer_quality,retrieval_ms,generation_s,answer,notes
0,small,fixed,Q1,fixed-28['kyoto-1'] 0.74 | fixed-31['osaka-1'] 0.64 | fixed-23['tokyo-1'] 0.52,1,10.0,10.0,23.9,5.47,"To get from the airport to Kyoto, you can either take the train or flight. For flights...",
1,small,fixed,Q2,fixed-36['lisbon-3'] 0.62 | fixed-39['porto-3'] 0.52 | fixed-34['lisbon-1'] 0.46,1,10.0,1.0,14.1,3.12,"In Lisbon, you should try the pastéis de Belém, a custard tart, as well as the frances...",answer dropped facts that were in the context
2,small,fixed,Q3,"fixed-19['venice-3'] 0.68 | fixed-16['venice-1'] 0.46 | fixed-10['rome-2', 'rome-3'] 0.40",1,10.0,5.5,5.4,3.44,"Venice may experience flooding due to high tides called acqua alta, especially during ...",answer dropped facts that were in the context
3,small,fixed,Q4,"fixed-21['milan-2', 'milan-3'] 0.55 | fixed-22['milan-3'] 0.39 | fixed-9['rome-1', 'ro...",1,10.0,5.5,16.0,1.92,"Yes, Leonardo da Vinci's Last Supper is recommended to visit in short timed slots in s...",answer dropped facts that were in the context
4,small,fixed,Q5,fixed-42['reykjavik-3'] 0.77 | fixed-40['reykjavik-1'] 0.42 | fixed-41['reykjavik-2'] ...,1,10.0,10.0,16.2,2.38,"No, it is highly unlikely to see the northern lights during a July trip to Reykjavik d...",
5,small,fixed,Q6,"fixed-4['new-york-1'] 0.61 | fixed-7['new-york-4'] 0.52 | fixed-6['new-york-2', 'new-y...",1,10.0,1.0,11.3,3.11,One cheap way to ride the subway a lot in New York is by purchasing an Unlimited Ride ...,"answer dropped facts that were in the context; unsupported numbers 127, 30, 33, 7"
6,small,fixed,Q7,fixed-36['lisbon-3'] 0.63 | fixed-39['porto-3'] 0.60 | fixed-34['lisbon-1'] 0.51,1,10.0,10.0,10.9,2.68,"Based on the provided information, Lisbon is cheaper per day than Porto. According to ...",
7,small,fixed,Q8,"fixed-33['osaka-3'] 0.57 | fixed-31['osaka-1'] 0.43 | fixed-26['tokyo-3', 'tokyo-4'] 0.39",1,10.0,5.5,13.0,2.16,"On a day trip from Osaka, you can see free-roaming deer in Nara, which is easily acces...",answer dropped facts that were in the context
8,small,overlap,Q1,"overlap-17['kyoto-1', 'kyoto-2'] 0.71 | overlap-14['tokyo-1', 'tokyo-2'] 0.52 | overla...",1,10.0,10.0,11.3,5.41,"To get from the airport to Kyoto, you can take the JR Haruka limited express train fro...",
9,small,overlap,Q2,"overlap-22['lisbon-2', 'lisbon-3'] 0.65 | overlap-24['porto-2', 'porto-3'] 0.55 | over...",1,10.0,1.0,5.1,5.02,"In Lisbon, you should try the bica de caldo verde, which is a type of black pudding ma...",answer dropped facts that were in the context


In [10]:
# Retrieved chunks for every row, in full (the brief asks for the chunks, not just the answers)
for _, r in results.iterrows():
    print(f"=== {r.embedding} / {r.chunking} / {r.qid}: {r.query}")
    for piece in r.retrieved_text.split(' ||| '):
        print('   ', piece[:600])
    print('   ANSWER:', r.answer.replace('\n', ' '))

=== small / fixed / Q1: How do I get from the airport to Kyoto?
    [fixed-28] Kyoto has no airport of its own. International travelers usually fly into Kansai International Airport (KIX) near Osaka and take the JR Haruka limited express, which reaches Kyoto Station in about 75 to 80 minutes. From Tokyo, the shinkansen bullet train takes about 2 hours 15 minutes. Inside the city, buses reach most temples but are often crowded; the two subway lines and the private Keihan and Hankyu railways are faster for some trips. Renting a bicycle works well because central Kyoto is flat. IC cards such as Suica and ICOCA are accepted on buses and trains.
    [fixed-31] Osaka is served by Kansai International Airport (KIX), built on an island in Osaka Bay. The Nankai Rapi:t train reaches Namba in about 35 to 40 minutes, and the JR Haruka goes to Tennoji and on to Kyoto. Within the city, the Metro and JR Osaka Loop Line cover the main districts, and IC cards such as ICOCA or Suica work on both. Osaka 

### Step 2.3 and 2.4: Comparing embedding models and chunking strategies

In [11]:
summary_cols = dict(gold_hit_rate=('gold_hit', 'mean'), mean_gold_rank=('gold_rank', 'mean'),
                    recall=('recall', 'mean'), context_quality=('context_quality', 'mean'),
                    answer_quality=('answer_quality', 'mean'), retrieval_ms=('retrieval_ms', 'mean'),
                    generation_s=('generation_s', 'mean'), prompt_tokens=('prompt_tokens', 'mean'))
by_config = results.groupby(['embedding', 'chunking'], sort=False).agg(**summary_cols).round(2)
by_embedding = results.groupby('embedding', sort=False).agg(**summary_cols).round(2)
by_chunking = results.groupby('chunking', sort=False).agg(**summary_cols).round(2)
for name, table in [('by_config', by_config), ('by_embedding', by_embedding), ('by_chunking', by_chunking)]:
    table.to_csv(RESULTS_DIR / f'summary_{name}.csv')
print('Per configuration'); display(by_config)
print('Per embedding model (averaged over the three chunking strategies)'); display(by_embedding)
print('Per chunking strategy (averaged over the three embedding models)'); display(by_chunking)
print('Answer quality, embedding x chunking'); display(results.pivot_table(index='embedding', columns='chunking',
      values='answer_quality', aggfunc='mean', sort=False).round(2))
print('Gold hits per query (out of 9 configurations)'); display(results.groupby('qid').gold_hit.sum())

Per configuration


gold_hit_rate  mean_gold_rank  recall  context_quality  answer_quality  retrieval_ms  generation_s  prompt_tokens
embedding chunking                                                                                                                   
small     fixed               1.0            1.00     1.0             10.0            6.06         13.85          3.04         466.00
          overlap             1.0            1.12     1.0             10.0            6.62          6.45          3.97         859.25
          hybrid              1.0            1.00     1.0             10.0            5.50          8.48          2.42         501.75
medium    fixed               1.0            1.00     1.0             10.0            7.75         17.51          2.84         449.88
          overlap             1.0            1.00     1.0             10.0            7.19         10.29          4.01         845.38
          hybrid              1.0            1.12     1.0             10.0            6.62         18.66          2.60         504.88
large     fixed               1.0            1.00     1.0             10.0            8.88         36.22          2.88         447.62
          overlap             1.0            1.00     1.0             10.0            7.75         30.44          3.74         847.75
          hybrid              1.0            1.00     1.0             10.0            5.50         26.86          2.26         504.62

Per embedding model (averaged over the three chunking strategies)


,gold_hit_rate,mean_gold_rank,recall,context_quality,answer_quality,retrieval_ms,generation_s,prompt_tokens
embedding,,,,,,,,
small,1.0,1.04,1.0,10.0,6.06,9.59,3.14,609.00
medium,1.0,1.04,1.0,10.0,7.19,15.49,3.15,600.04
large,1.0,1.00,1.0,10.0,7.38,31.18,2.96,600.00


Per chunking strategy (averaged over the three embedding models)


,gold_hit_rate,mean_gold_rank,recall,context_quality,answer_quality,retrieval_ms,generation_s,prompt_tokens
chunking,,,,,,,,
fixed,1.0,1.00,1.0,10.0,7.56,22.53,2.92,454.50
overlap,1.0,1.04,1.0,10.0,7.19,15.72,3.91,850.79
hybrid,1.0,1.04,1.0,10.0,5.88,18.00,2.43,503.75


Answer quality, embedding x chunking


chunking,fixed,overlap,hybrid
embedding,,,
small,6.06,6.62,5.50
medium,7.75,7.19,6.62
large,8.88,7.75,5.50


Gold hits per query (out of 9 configurations)


qid
Q1    9
Q2    9
Q3    9
Q4    9
Q5    9
Q6    9
Q7    9
Q8    9
Name: gold_hit, dtype: int64

In [12]:
# Retrieval only, at more depths: how far down the ranking is the gold paragraph?
def first_gold_rank(store, q, depth=10):
    hits, _ = store.search(q['query'], k=depth)
    ranks = [h['rank'] for h in hits if set(h['pids']) & set(q['gold'])]
    return min(ranks) if ranks else None

depth_rows = []
for (size, strategy), store in STORES.items():
    ranks = [first_gold_rank(store, q) for q in QUERIES]
    depth_rows.append(dict(embedding=size, chunking=strategy,
                           **{f'hit@{k}': sum(r is not None and r <= k for r in ranks) for k in (1, 3, 5)},
                           ranks=' '.join(str(r) for r in ranks)))
depth_table = pd.DataFrame(depth_rows)
depth_table.to_csv(RESULTS_DIR / 'retrieval_depth.csv', index=False)
display(depth_table)

,embedding,chunking,hit@1,hit@3,hit@5,ranks
0,small,fixed,8,8,8,1 1 1 1 1 1 1 1
1,small,overlap,7,8,8,1 1 2 1 1 1 1 1
2,small,hybrid,8,8,8,1 1 1 1 1 1 1 1
3,medium,fixed,8,8,8,1 1 1 1 1 1 1 1
4,medium,overlap,8,8,8,1 1 1 1 1 1 1 1
5,medium,hybrid,7,8,8,1 1 1 2 1 1 1 1
6,large,fixed,8,8,8,1 1 1 1 1 1 1 1
7,large,overlap,8,8,8,1 1 1 1 1 1 1 1
8,large,hybrid,8,8,8,1 1 1 1 1 1 1 1


### Manual review of all 72 answers

Retrieval found the gold paragraph in every row, so the script's context score is 10 everywhere and its answer score
only checks for key words. Reading the answers showed that is not enough: on Q7 every answer contains "85" and "95"
and scores 10, but six of the nine say Lisbon is the cheaper city. So we read all 72 answers and labelled each one:

* **correct**: answers the question with facts from the retrieved chunks only;
* **partial**: the core answer is right but it adds something wrong or not in the guides (a hallucinated detail);
* **wrong**: the answer is wrong, contradicts the context, or is built on invented facts.

The labels and a one-line reason for each are in `results/manual_review.csv`; the cell below summarises them
(correct = 1 point, partial = 0.5, wrong = 0, out of 8 per configuration).

In [13]:
POINTS = {'correct': 1, 'partial': 0.5, 'wrong': 0}

def load_review(name, frame, keys):
    path = RESULTS_DIR / name
    if not path.exists():
        print(f'{name} not found: the manual review is written after reading a run.')
        return None
    review = pd.read_csv(path)
    merged = frame.merge(review, on=keys, how='left')
    stale = merged[merged.answer.str[:60] != merged.answer_start]
    print(f'{len(review)} reviewed answers; {len(stale)} differ from this run' + (' (re-review needed)' if len(stale) else ''))
    merged['points'] = merged.verdict.map(POINTS)
    return merged

reviewed = load_review('manual_review.csv', results, ['embedding', 'chunking', 'qid'])
if reviewed is not None:
    print('Points out of 8, embedding x chunking'); display(reviewed.pivot_table(index='embedding', columns='chunking',
          values='points', aggfunc='sum', sort=False))
    print('Verdicts per query'); display(pd.crosstab(reviewed.qid, reviewed.verdict))
    print('Verdicts per configuration'); display(pd.crosstab([reviewed.embedding, reviewed.chunking], reviewed.verdict))

72 reviewed answers; 0 differ from this run
Points out of 8, embedding x chunking


chunking,fixed,overlap,hybrid
embedding,,,
small,3.5,3.0,3.5
medium,5.5,3.5,5.5
large,6.5,3.0,5.0


Verdicts per query


verdict,correct,partial,wrong
qid,,,
Q1,2,7,0
Q2,3,4,2
Q3,3,3,3
Q4,8,1,0
Q5,0,9,0
Q6,1,1,7
Q7,3,0,6
Q8,4,5,0


Verdicts per configuration


verdict             correct  partial  wrong
embedding chunking                         
large     fixed           5        3      0
          hybrid          4        2      2
          overlap         0        6      2
medium    fixed           5        1      2
          hybrid          4        3      1
          overlap         2        3      3
small     fixed           1        5      2
          hybrid          2        3      3
          overlap         1        4      3

### Step 2.5: Data scaling

The same eight queries are run against three knowledge bases, using the configuration with the best mean answer
quality from Step 2.2 (picked by the code below, not by hand):

* **Small**: a subset of five cities (Tokyo, Kyoto, Osaka, Lisbon, Porto). Only Q1, Q2, Q7 and Q8 can be answered
  from it; the other four test what the system does when the answer is not there.
* **Base**: the 40-paragraph knowledge base used above.
* **Large**: base plus the 19 extra paragraphs (TakeOff help pages and general travel advice, including near-topic
  text such as rail passes in Japan and trains in Italy).

Retrieval is also measured for all three embedding models at each size, without generation. **Noise** is counted as
retrieved chunks that come from documents other than the gold one, and the **margin** is the similarity of the best
gold chunk minus the best non-gold chunk (positive means the right chunk wins).

In [14]:
best = by_config.sort_values(['answer_quality', 'context_quality', 'gold_hit_rate'], ascending=False).index[0]
BEST_SIZE, BEST_STRATEGY = best
print('Best configuration from Step 2.2:', BEST_SIZE, '/', BEST_STRATEGY)

SUBSET_CITIES = {'Tokyo, Japan', 'Kyoto, Japan', 'Osaka, Japan', 'Lisbon, Portugal', 'Porto, Portugal'}
DATASETS = {'small': [p for p in PARAGRAPHS if p['doc'] in SUBSET_CITIES], 'base': PARAGRAPHS, 'large': PARAGRAPHS + EXTRA}
for name, paras in DATASETS.items():
    print(f"{name:>5}: {len(paras)} paragraphs, {sum(p['words'] for p in paras)} words")

def gold_docs(q):
    return {PARA_BY_ID[g]['doc'] for g in q['gold']}

scale_retrieval, scale_rows = [], []
for dname, paras in DATASETS.items():
    chunks_by_strategy = build_chunks(paras)
    for size in EMBEDDING_MODELS:
        store = VectorStore(EMBEDDERS[size], size, chunks_by_strategy[BEST_STRATEGY])
        for q in QUERIES:
            answerable = all(g in {p['pid'] for p in paras} for g in q['gold'])
            hits, _ = store.search(q['query'], k=TOP_K)
            gold_scores = [h['score'] for h in hits if set(h['pids']) & set(q['gold'])]
            all_hits, _ = store.search(q['query'], k=len(store.chunks))
            best_gold = max([h['score'] for h in all_hits if set(h['pids']) & set(q['gold'])], default=None)
            best_other = max([h['score'] for h in all_hits if not set(h['pids']) & set(q['gold'])], default=None)
            scale_retrieval.append(dict(dataset=dname, embedding=size, qid=q['qid'], answerable=answerable,
                gold_hit=bool(gold_scores), top1_score=round(hits[0]['score'], 3),
                noise_chunks=sum(h['doc'] not in gold_docs(q) for h in hits),
                extra_doc_chunks=sum(h['doc'] not in {p['doc'] for p in PARAGRAPHS} for h in hits),
                margin=round(best_gold - best_other, 3) if best_gold is not None and best_other is not None else None))
            if size == BEST_SIZE:
                res = answer(q['query'], store)
                scale_rows.append(dict(evaluate(q, res, size, BEST_STRATEGY, label=f'scale-{dname}'),
                                       dataset=dname, answerable=answerable))
scale_retrieval = pd.DataFrame(scale_retrieval)
scale_results = pd.DataFrame(scale_rows)
scale_retrieval.to_csv(RESULTS_DIR / 'scaling_retrieval.csv', index=False)
scale_results.to_csv(RESULTS_DIR / 'scaling_answers.csv', index=False)

print('\nRetrieval by dataset size (all three embedding models)')
display(scale_retrieval.groupby(['dataset', 'embedding'], sort=False).agg(
    gold_hit_rate=('gold_hit', 'mean'), noise_chunks=('noise_chunks', 'mean'),
    extra_doc_chunks=('extra_doc_chunks', 'mean'), margin=('margin', 'mean'), top1_score=('top1_score', 'mean')).round(3))
print(f'Answers by dataset size ({BEST_SIZE} / {BEST_STRATEGY})')
display(scale_results.groupby(['dataset', 'answerable'], sort=False).agg(
    queries=('qid', 'count'), gold_hit_rate=('gold_hit', 'mean'), context_quality=('context_quality', 'mean'),
    answer_quality=('answer_quality', 'mean')).round(2))
with pd.option_context('display.max_colwidth', 110):
    display(scale_results[['dataset', 'qid', 'answerable', 'retrieved', 'answer_quality', 'answer', 'notes']])

Best configuration from Step 2.2: large / fixed
small: 16 paragraphs, 1445 words
 base: 40 paragraphs, 3732 words
large: 59 paragraphs, 5441 words

Retrieval by dataset size (all three embedding models)


gold_hit_rate  noise_chunks  extra_doc_chunks  margin  top1_score
dataset embedding                                                                   
small   small                0.5         2.000             0.000   0.116       0.457
        medium               0.5         2.000             0.000   0.087       0.451
        large                0.5         1.875             0.000   0.101       0.636
base    small                1.0         0.750             0.000   0.159       0.646
        medium               1.0         0.625             0.000   0.147       0.641
        large                1.0         0.625             0.000   0.099       0.725
large   small                1.0         0.750             0.000   0.159       0.646
        medium               1.0         0.625             0.000   0.147       0.641
        large                1.0         0.750             0.125   0.099       0.725

Answers by dataset size (large / fixed)


queries  gold_hit_rate  context_quality  answer_quality
dataset answerable                                                         
small   True              4            1.0            10.00            8.88
        False             4            0.0             2.12            4.38
base    True              8            1.0            10.00            8.88
large   True              8            1.0            10.00            8.88

,dataset,qid,answerable,retrieved,answer_quality,answer,notes
0,small,Q1,True,fixed-5['kyoto-1'] 0.81 | fixed-8['osaka-1'] 0.67 | fixed-7['kyoto-3'] 0.64,10.0,"To get from the airport to Kyoto, you can either take the train or bus. From Kansai International Airport ...",
1,small,Q2,True,fixed-13['lisbon-3'] 0.72 | fixed-12['lisbon-2'] 0.64 | fixed-16['porto-3'] 0.61,5.5,"In Lisbon, you should try the Pastéis de Belém, a custard tart, as it is Lisbon's signature pastry. Additi...",answer dropped facts that were in the context
2,small,Q3,False,"fixed-3['tokyo-3', 'tokyo-4'] 0.55 | fixed-7['kyoto-3'] 0.52 | fixed-13['lisbon-3'] 0.47",5.5,"Venice is known for being prone to flooding due to high tides called ""acqua alta,"" which typically occur d...",gold paragraph not retrieved
3,small,Q4,False,"fixed-3['tokyo-3', 'tokyo-4'] 0.53 | fixed-2['tokyo-2', 'tokyo-3'] 0.51 | fixed-6['kyoto-2', 'kyoto-3'] 0.50",5.5,"No, there is no need to book the Last Supper in Milan ahead of time. It is an open restaurant where custom...",gold paragraph not retrieved
4,small,Q5,False,fixed-13['lisbon-3'] 0.53 | fixed-10['osaka-3'] 0.48 | fixed-0['tokyo-1'] 0.46,5.5,"No, you cannot see the Northern Lights (Aurora Borealis) in Reykjavik during the summer months, which typi...",gold paragraph not retrieved; unsupported numbers 24
5,small,Q6,False,fixed-11['lisbon-1'] 0.55 | fixed-0['tokyo-1'] 0.55 | fixed-14['porto-1'] 0.51,1.0,"In New York City, a cheap way to ride the subway a lot is by purchasing a MetroCard, which can be refilled...",gold paragraph not retrieved
6,small,Q7,True,fixed-16['porto-3'] 0.68 | fixed-13['lisbon-3'] 0.68 | fixed-14['porto-1'] 0.63,10.0,"Based on the given information, Porto is cheaper per day compared to Lisbon. According to TakeOff's destin...",
7,small,Q8,True,"fixed-10['osaka-3'] 0.71 | fixed-8['osaka-1'] 0.58 | fixed-6['kyoto-2', 'kyoto-3'] 0.58",10.0,"On a day trip from Osaka, you can visit Nara, which is easily accessible via the Nankai Rapi:t train or th...",
8,base,Q1,True,fixed-28['kyoto-1'] 0.81 | fixed-31['osaka-1'] 0.67 | fixed-30['kyoto-3'] 0.64,10.0,"To get from the airport to Kyoto, you can either take the train or bus. From Kansai International Airport ...",
9,base,Q2,True,"fixed-36['lisbon-3'] 0.72 | fixed-35['lisbon-2'] 0.64 | fixed-18['venice-2', 'venice-3'] 0.62",5.5,"In Lisbon, you should try the local food such as Pastéis de Belém (custard tarts), bifanas (pork sandwiche...",answer dropped facts that were in the context


## Part 3: Failure analysis and improvement

### Step 3.1: Failure candidates

The code lists every row where retrieval missed the gold paragraph, where the answer lost facts that were in the
context, or where the answer contains numbers that the context does not. The failures discussed in
`rag_analysis.md` are taken from this list, with their retrieved chunks shown in full.

In [15]:
failures = results[(~results.gold_hit) | (results.answer_quality < results.context_quality) |
                   (results.unsupported_numbers != '')]
print(f'{len(failures)} of {len(results)} rows flagged')
print('Flagged rows per query:'); display(failures.groupby('qid').size())
for _, r in failures.iterrows():
    print(f"--- {r.embedding}/{r.chunking}/{r.qid} | ctx {r.context_quality} ans {r.answer_quality} | {r.notes}")
    print('    retrieved:', r.retrieved)
    print('    answer   :', r.answer.replace('\n', ' ')[:400])

36 of 72 rows flagged
Flagged rows per query:


qid
Q1    1
Q2    9
Q3    6
Q4    7
Q5    3
Q6    8
Q8    2
dtype: int64

--- small/fixed/Q2 | ctx 10.0 ans 1.0 | answer dropped facts that were in the context
    retrieved: fixed-36['lisbon-3'] 0.62 | fixed-39['porto-3'] 0.52 | fixed-34['lisbon-1'] 0.46
    answer   : In Lisbon, you should try the pastéis de Belém, a custard tart, as well as the francesinha, a sandwich layered with meats, covered in melted cheese and a beer-based sauce. Additionally, consider trying some funicular rides to explore the city's steep hills.
--- small/fixed/Q3 | ctx 10.0 ans 5.5 | answer dropped facts that were in the context
    retrieved: fixed-19['venice-3'] 0.68 | fixed-16['venice-1'] 0.46 | fixed-10['rome-2', 'rome-3'] 0.40
    answer   : Venice may experience flooding due to high tides called acqua alta, especially during peak tourist season (autumn to early spring). However, this typically occurs less frequently in November compared to other months like September and October when the city experiences more frequent and severe floods. Therefore, it is possible but not gua

### Step 3.3, round 1: general fixes (before and after)

We first tried three general fixes from the brief's list, chosen before the root-cause analysis, each changing one
thing and run on all eight queries:

1. **Refine the query**: when the question names one of TakeOff's cities, add the city and country in the same
   form as the hybrid headers ("Kyoto, Japan"), so short questions point at the right document.
2. **Change top-k**: 3 → 5, for questions that need two documents (Q7).
3. **Improve the prompt**: number the sources, ask the model to copy figures exactly, answer yes or no first when
   the question is a yes/no question, and state which city a fact belongs to.

They are run on the best configuration and on the weakest one, because a fix that only helps a weak setup tells
us something different from one that also helps the best.

In [16]:
CITY_NAMES = {p['doc'].split(',')[0].lower(): p['doc'] for p in PARAGRAPHS}

def city_rewrite(query):
    found = [full for city, full in CITY_NAMES.items() if city in query.lower()]
    return (' '.join(found) + ': ' + query) if found else query

IMPROVED_PROMPT = ("You are TakeOff's travel assistant. Use only the numbered context below. "
                   "If the question can be answered with yes or no, start with Yes or No. "
                   "Copy any number, price or time exactly as written in the context and say which city it is for. "
                   "If the context does not contain the answer, say that you do not know. "
                   "Answer in two or three sentences.")

worst = by_config.sort_values(['answer_quality', 'context_quality']).index[0]
VARIANTS = {
    'before (baseline)': dict(),
    'fix 1: city in query': dict(rewrite=city_rewrite),
    'fix 2: top-k 5': dict(k=5),
    'fix 3: improved prompt': dict(system=IMPROVED_PROMPT),
}
improve_rows = []
for size, strategy in dict.fromkeys([best, worst]):
    store = STORES[(size, strategy)]
    for label, kwargs in VARIANTS.items():
        for q in QUERIES:
            res = answer(q['query'], store, **kwargs)
            improve_rows.append(evaluate(q, res, size, strategy, k=kwargs.get('k', TOP_K), label=label))
improvements = pd.DataFrame(improve_rows)
improvements.to_csv(RESULTS_DIR / 'improvements.csv', index=False)
print('best:', best, '| weakest:', worst)
display(improvements.groupby(['embedding', 'chunking', 'run'], sort=False).agg(
    gold_hit_rate=('gold_hit', 'mean'), context_quality=('context_quality', 'mean'),
    answer_quality=('answer_quality', 'mean'), unsupported=('unsupported_numbers', lambda s: (s != '').sum()),
    generation_s=('generation_s', 'mean'), prompt_tokens=('prompt_tokens', 'mean')).round(2))
print('Per query, answer quality (before -> each fix)')
display(improvements.pivot_table(index=['embedding', 'chunking', 'qid'], columns='run', values='answer_quality',
                                 sort=False)[list(VARIANTS)])

best: ('large', 'fixed') | weakest: ('small', 'hybrid')


gold_hit_rate  context_quality  answer_quality  unsupported  generation_s  prompt_tokens
embedding chunking run                                                                                                             
large     fixed    before (baseline)                 1.0             10.0            8.88            0          2.98         447.62
                   fix 1: city in query              1.0             10.0            7.75            1          2.93         451.88
                   fix 2: top-k 5                    1.0             10.0            8.31            1          3.31         710.75
                   fix 3: improved prompt            1.0             10.0            7.56            0          2.83         481.62
small     hybrid   before (baseline)                 1.0             10.0            5.50            1          2.42         501.75
                   fix 1: city in query              1.0             10.0            5.50            1          2.49         505.00
                   fix 2: top-k 5                    1.0             10.0            7.75            2          3.64         787.88
                   fix 3: improved prompt            1.0             10.0            6.62            1          2.55         535.75

Per query, answer quality (before -> each fix)


run                     before (baseline)  fix 1: city in query  fix 2: top-k 5  fix 3: improved prompt
embedding chunking qid                                                                                 
large     fixed    Q1                10.0                  10.0            10.0                    10.0
                   Q2                 5.5                   5.5             5.5                     5.5
                   Q3                10.0                  10.0            10.0                     5.5
                   Q4                 5.5                   5.5             5.5                     5.5
                   Q5                10.0                  10.0            10.0                    10.0
                   Q6                10.0                   1.0             5.5                    10.0
                   Q7                10.0                  10.0            10.0                     4.0
                   Q8                10.0                  10.0            10.0                    10.0
small     hybrid   Q1                10.0                  10.0            10.0                    10.0
                   Q2                 1.0                   1.0             1.0                     1.0
                   Q3                 1.0                   1.0            10.0                     1.0
                   Q4                 5.5                   5.5            10.0                    10.0
                   Q5                 5.5                   5.5            10.0                     5.5
                   Q6                 1.0                   1.0             1.0                     5.5
                   Q7                10.0                  10.0            10.0                    10.0
                   Q8                10.0                  10.0            10.0                    10.0

In [17]:
# Answers before and after, side by side, for every query and configuration
for (size, strategy), grp in improvements.groupby(['embedding', 'chunking'], sort=False):
    for qid, g in grp.groupby('qid', sort=False):
        print(f"=== {size}/{strategy}/{qid}: {g.iloc[0].query}")
        for _, r in g.iterrows():
            print(f"  [{r.run}] ans={r.answer_quality} ctx={r.context_quality} | {r.retrieved}")
            print('     ', r.answer.replace('\n', ' ')[:350])

=== large/fixed/Q1: How do I get from the airport to Kyoto?
  [before (baseline)] ans=10.0 ctx=10.0 | fixed-28['kyoto-1'] 0.81 | fixed-31['osaka-1'] 0.67 | fixed-30['kyoto-3'] 0.64
      To get from the airport to Kyoto, you can either take the train or bus. From Kansai International Airport (KIX), you can take the JR Haruka limited express train to Kyoto Station, which takes about 75-80 minutes. Alternatively, you can also use the shinkansen bullet train from Tokyo, taking around 2 hours 15 minutes. Within the city, buses and subw
  [fix 1: city in query] ans=10.0 ctx=10.0 | fixed-28['kyoto-1'] 0.80 | fixed-31['osaka-1'] 0.66 | fixed-30['kyoto-3'] 0.63
      To get from the airport to Kyoto, you can either take the train or bus. From Kansai International Airport (KIX), you can take the JR Haruka limited express train to Kyoto Station, which takes about 75-80 minutes. Alternatively, you can also use the shinkansen bullet train from Tokyo, taking around 2 hours 15 minutes. Within the ci

### Step 3.3, round 2: fixes aimed at the root causes

Round 1 showed that general fixes move answers around without fixing them (see `rag_analysis.md`). The root-cause
analysis pointed at three specific causes, so round 2 tests one fix per cause, then all three together:

| Fix | Root cause it targets | Change |
|---|---|---|
| **A. 60-word chunks** | chunking: in the Venice guide the flooding fact sits behind three food sentences in one paragraph headed "Food, seasons and costs", and hybrid chunks keep it there (Q3) | chunks of at most 60 words, still whole sentences and still with the city/topic header |
| **B. City filter** | retrieval: slots 2-3 are filled with a neighbouring city on the same topic, and the model borrows its facts (Porto's francesinha "in Lisbon", Q2) | when the question names a city, only that city's chunks are searched |
| **C. Grounded prompt, best passage last** | generation: the model adds facts it already "knows" (MetroCard prices, taxis, car hire; Q1, Q6) | a stricter instruction ("do not add prices, times, places or tips that are not in the context") and the best passage placed directly before the question |
| **A+B+C** | all of the above | the design used in the TakeOff app (A9) |

Run on the best and weakest configurations from Step 2.2 and on medium/hybrid, the candidate for the app.

In [18]:
def chunk_hybrid_small(paragraphs, max_words=60):
    # Hybrid chunks, but paragraphs are cut into runs of whole sentences of at most max_words words.
    chunks = []
    for p in paragraphs:
        header, current = f"{p['doc']} / {p['topic']}: ", []
        for s in sentences(p['text']):
            if current and len(' '.join(current + [s]).split()) > max_words:
                chunks.append(dict(text=header + ' '.join(current), doc=p['doc'], pids=[p['pid']]))
                current = []
            current.append(s)
        if current:
            chunks.append(dict(text=header + ' '.join(current), doc=p['doc'], pids=[p['pid']]))
    return chunks

SMALL_CHUNKERS = {'fixed': lambda ps: chunk_fixed(ps, max_words=60), 'hybrid': chunk_hybrid_small,
                  'overlap': lambda ps: chunk_fixed(ps, max_words=60)}
_small_stores = {}

def small_store(size, strategy):
    if (size, strategy) not in _small_stores:
        chunks = [dict(c, cid=f'{strategy}60-{i}', words=len(c['text'].split()))
                  for i, c in enumerate(SMALL_CHUNKERS[strategy](PARAGRAPHS))]
        check_chunks(chunks, PARAGRAPHS)
        _small_stores[(size, strategy)] = VectorStore(EMBEDDERS[size], size, chunks)
    return _small_stores[(size, strategy)]

def cities_named(query):
    found = {full for city, full in CITY_NAMES.items() if city in query.lower()}
    return found or None

GROUNDED_PROMPT = ("You are TakeOff's travel assistant. Answer the traveler's question using only facts stated in "
                   "the numbered context below. Do not add prices, times, places or tips that are not in the context. "
                   "Each passage starts with the city it is about; use only passages about the city in the question. "
                   "If the context does not contain the answer, say that you do not know. "
                   "Answer in two or three sentences.")

ROUND2 = {
    'before': dict(),
    'A: 60-word chunks': dict(small=True),
    'B: city filter': dict(city_filter=True),
    'C: grounded prompt, best last': dict(system=GROUNDED_PROMPT, best_last=True),
    'A+B+C': dict(small=True, city_filter=True, system=GROUNDED_PROMPT, best_last=True),
}
CONFIGS2 = list(dict.fromkeys([best, ('medium', 'hybrid'), worst]))
round2_rows = []
for size, strategy in CONFIGS2:
    for label, opts in ROUND2.items():
        store = small_store(size, strategy) if opts.get('small') else STORES[(size, strategy)]
        for q in QUERIES:
            res = answer(q['query'], store, system=opts.get('system', SYSTEM_PROMPT), best_last=opts.get('best_last', False),
                         docs=cities_named(q['query']) if opts.get('city_filter') else None)
            round2_rows.append(dict(evaluate(q, res, size, strategy, label=label), chunks_in_store=len(store.chunks)))
    print(f'{size}/{strategy} done')
round2 = pd.DataFrame(round2_rows)
round2.to_csv(RESULTS_DIR / 'improvements_round2.csv', index=False)
print('chunk counts with 60-word chunks:', {k: len(v.chunks) for k, v in _small_stores.items()})
display(round2.groupby(['embedding', 'chunking', 'run'], sort=False).agg(
    answer_quality=('answer_quality', 'mean'), unsupported=('unsupported_numbers', lambda s: (s != '').sum()),
    generation_s=('generation_s', 'mean'), prompt_tokens=('prompt_tokens', 'mean')).round(2))

large/fixed done
medium/hybrid done
small/hybrid done
chunk counts with 60-word chunks: {('large', 'fixed'): 79, ('medium', 'hybrid'): 86, ('small', 'hybrid'): 86}


answer_quality  unsupported  generation_s  prompt_tokens
embedding chunking run                                                                                    
large     fixed    before                                   8.88            0          2.97         447.62
                   A: 60-word chunks                        8.88            0          2.40         288.88
                   B: city filter                           8.31            1          2.90         461.38
                   C: grounded prompt, best last            7.00            0          2.95         488.62
                   A+B+C                                    7.75            1          2.48         331.25
medium    hybrid   before                                   6.62            1          2.57         504.88
                   A: 60-word chunks                        6.62            1          2.46         299.88
                   B: city filter                           6.06            1          2.93         503.38
                   C: grounded prompt, best last            7.75            1          3.01         545.88
                   A+B+C                                    7.19            2          2.52         347.25
small     hybrid   before                                   5.50            1          2.50         501.75
                   A: 60-word chunks                        7.75            1          2.47         303.00
                   B: city filter                           6.06            1          2.93         506.00
                   C: grounded prompt, best last            6.62            1          2.80         542.75
                   A+B+C                                    7.19            1          2.47         346.12

In [19]:
# Round 2 answers side by side
for (size, strategy), grp in round2.groupby(['embedding', 'chunking'], sort=False):
    for qid, g in grp.groupby('qid', sort=False):
        print(f"=== {size}/{strategy}/{qid}: {g.iloc[0].query}")
        for _, r in g.iterrows():
            print(f"  [{r.run}] {r.retrieved}")
            print('     ', r.answer.replace('\n', ' ')[:350])

reviewed2 = load_review('manual_review_round2.csv', round2, ['embedding', 'chunking', 'qid', 'run'])
if reviewed2 is not None:
    print('Manual review, points out of 8'); display(reviewed2.pivot_table(index=['embedding', 'chunking'],
          columns='run', values='points', aggfunc='sum', sort=False)[list(ROUND2)])

=== large/fixed/Q1: How do I get from the airport to Kyoto?
  [before] fixed-28['kyoto-1'] 0.81 | fixed-31['osaka-1'] 0.67 | fixed-30['kyoto-3'] 0.64
      To get from the airport to Kyoto, you can either take the train or bus. From Kansai International Airport (KIX), you can take the JR Haruka limited express train to Kyoto Station, which takes about 75-80 minutes. Alternatively, you can also use the shinkansen bullet train from Tokyo, taking around 2 hours 15 minutes. Within the city, buses and subw
  [A: 60-word chunks] fixed60-48['kyoto-1'] 0.81 | fixed60-49['kyoto-1'] 0.71 | fixed60-54['osaka-1'] 0.66
      To get from the airport to Kyoto, you can take the JR Haruka limited express train from Kansai International Airport (KIX) to Kyoto Station. This journey typically takes around 75-80 minutes. Alternatively, you can also use the shinkansen bullet train from Tokyo, taking approximately 2 hours and 15 minutes.
  [B: city filter] fixed-28['kyoto-1'] 0.81 | fixed-30['kyoto-3'] 0.64 

,run,before,A: 60-word chunks,B: city filter,"C: grounded prompt, best last",A+B+C
embedding,chunking,,,,,
large,fixed,6.5,5.0,6.0,3.5,4.5
medium,hybrid,5.5,4.5,4.5,3.5,6.5
small,hybrid,3.5,5.0,3.5,4.0,6.0


### Calibrating a "no match" threshold for the app

In the scaling test the model answered questions whose answer was not in the knowledge base (and invented facts
while doing so), even though the prompt tells it to say it does not know. A retrieval score threshold stops those
questions before they reach the model. To choose it, we search each city's own chunks (as the app does, filtered to
the trip's cities) with the eight answerable questions and with eight questions the guides do not answer, and compare
the best scores.

In [20]:
OFF_TOPIC = [
    ('What is the best sushi restaurant in Tokyo for a birthday dinner?', {'Tokyo, Japan'}),
    ('Which hotel in Kyoto has the best spa?', {'Kyoto, Japan'}),
    ('Is there a good dentist near the Colosseum in Rome?', {'Rome, Italy'}),
    ('What is the wifi password at Lisbon airport?', {'Lisbon, Portugal'}),
    ('Can I bring my dog to a restaurant in Chicago?', {'Chicago, United States'}),
    ('Where can I buy hiking boots in Reykjavik?', {'Reykjavik, Iceland'}),
    ('Who designed the Milan metro map?', {'Milan, Italy'}),
    ('What time does the Porto cathedral open on Sundays?', {'Porto, Portugal'}),
]
cal_rows = []
for size in EMBEDDING_MODELS:
    for strategy, store in [('hybrid', STORES[(size, 'hybrid')]), ('hybrid60', small_store(size, 'hybrid'))]:
        for q in QUERIES:
            hits, _ = store.search(q['query'], k=1, docs=cities_named(q['query']))
            cal_rows.append(dict(embedding=size, chunking=strategy, kind='answerable', query=q['query'], top1=hits[0]['score']))
        for text, docs in OFF_TOPIC:
            hits, _ = store.search(text, k=1, docs=docs)
            cal_rows.append(dict(embedding=size, chunking=strategy, kind='not in guides', query=text, top1=hits[0]['score']))
calibration = pd.DataFrame(cal_rows)
calibration.to_csv(RESULTS_DIR / 'threshold_calibration.csv', index=False)
summary = calibration.groupby(['embedding', 'chunking', 'kind'], sort=False).top1.agg(['min', 'max', 'mean']).unstack('kind').round(3)
summary['gap'] = (summary[('min', 'answerable')] - summary[('max', 'not in guides')]).round(3)
summary['threshold'] = ((summary[('min', 'answerable')] + summary[('max', 'not in guides')]) / 2).round(2)
display(summary)
with pd.option_context('display.max_rows', 100):
    display(calibration.pivot_table(index=['kind', 'query'], columns=['embedding', 'chunking'], values='top1', sort=False).round(2))

min                      max                     mean                  gap threshold
kind               answerable not in guides answerable not in guides answerable not in guides                 
embedding chunking                                                                                            
small     hybrid        0.514         0.396      0.768         0.681      0.650         0.522 -0.167      0.60
          hybrid60      0.614         0.408      0.785         0.726      0.683         0.540 -0.112      0.67
medium    hybrid        0.457         0.354      0.797         0.609      0.647         0.472 -0.152      0.53
          hybrid60      0.551         0.390      0.771         0.620      0.677         0.470 -0.069      0.59
large     hybrid        0.645         0.552      0.805         0.702      0.709         0.612 -0.057      0.67
          hybrid60      0.712         0.557      0.808         0.681      0.732         0.613  0.031      0.70

embedding                                                                        small          medium           large         
chunking                                                                        hybrid hybrid60 hybrid hybrid60 hybrid hybrid60
kind          query                                                                                                            
answerable    How do I get from the airport to Kyoto?                             0.74     0.74   0.80     0.75   0.81     0.81
              What local food should I try in Lisbon?                             0.66     0.63   0.69     0.74   0.70     0.71
              Will Venice be flooded if I visit in November?                      0.60     0.68   0.59     0.69   0.65     0.72
              Do I need to book the Last Supper in Milan ahead of time?           0.51     0.64   0.46     0.61   0.64     0.72
              Can I see the northern lights on a July trip to Reykjavik?          0.77     0.77   0.78     0.77   0.75     0.73
              What is a cheap way to ride the subway a lot in New York?           0.64     0.62   0.69     0.62   0.73     0.71
              Which is cheaper per day, Lisbon or Porto?                          0.69     0.79   0.63     0.69   0.69     0.73
              Where can I see free-roaming deer on a day trip from Osaka?         0.58     0.61   0.54     0.55   0.71     0.73
not in guides What is the best sushi restaurant in Tokyo for a birthday dinner?   0.68     0.73   0.61     0.62   0.70     0.68
              Which hotel in Kyoto has the best spa?                              0.60     0.58   0.56     0.54   0.61     0.60
              Is there a good dentist near the Colosseum in Rome?                 0.52     0.51   0.35     0.46   0.63     0.65
              What is the wifi password at Lisbon airport?                        0.42     0.47   0.41     0.39   0.61     0.62
              Can I bring my dog to a restaurant in Chicago?                      0.58     0.60   0.49     0.44   0.60     0.60
              Where can I buy hiking boots in Reykjavik?                          0.46     0.48   0.41     0.40   0.55     0.56
              Who designed the Milan metro map?                                   0.52     0.54   0.50     0.48   0.58     0.58
              What time does the Porto cathedral open on Sundays?                 0.40     0.41   0.44     0.44   0.60     0.60

## Part 4 inputs: cost drivers

Measured numbers behind the cost discussion in `rag_analysis.md`: what each embedding model costs to store and
compute, how chunk size changes the prompt, and how top-k changes the prompt and generation time.

In [21]:
cost_rows = []
for (size, strategy), store in STORES.items():
    n, dim = store.matrix.shape
    for k in (1, 3, 5):
        probe = [answer(q['query'], store, k=k) for q in QUERIES[:3]] if (size, strategy) == best else None
        hits_tokens = [GENERATOR.count_tokens(build_messages(q['query'], store.search(q['query'], k=k)[0]))
                       for q in QUERIES]
        cost_rows.append(dict(embedding=size, chunking=strategy, k=k, dims=dim, chunks=n,
                              bytes_per_vector=dim * 4, index_kb=round(store.matrix.nbytes / 1024, 1),
                              embed_ms_per_chunk=round(1000 * store.build_seconds / n, 1),
                              mean_prompt_tokens=round(np.mean(hits_tokens)),
                              mean_generation_s=round(np.mean([p['generation_s'] for p in probe]), 2) if probe else None))
cost_table = pd.DataFrame(cost_rows)
cost_table.to_csv(RESULTS_DIR / 'cost_table.csv', index=False)
with pd.option_context('display.max_rows', 40):
    display(cost_table)

,embedding,chunking,k,dims,chunks,bytes_per_vector,index_kb,embed_ms_per_chunk,mean_prompt_tokens,mean_generation_s
0,small,fixed,1,384,44,1536,66.0,5.3,202,NaN
1,small,fixed,3,384,44,1536,66.0,5.3,466,NaN
2,small,fixed,5,384,44,1536,66.0,5.3,712,NaN
3,small,overlap,1,384,28,1536,42.0,3.9,335,NaN
4,small,overlap,3,384,28,1536,42.0,3.9,859,NaN
5,small,overlap,5,384,28,1536,42.0,3.9,1379,NaN
6,small,hybrid,1,384,40,1536,60.0,2.3,220,NaN
7,small,hybrid,3,384,40,1536,60.0,2.3,502,NaN
8,small,hybrid,5,384,40,1536,60.0,2.3,788,NaN
9,medium,fixed,1,768,44,3072,132.0,13.4,200,NaN


## System design

```
User → Query → Embedding model → Vector store (cosine top-k) → Retrieved context → Prompt → SmolLM2-1.7B → Response
```

Written answers for Steps 2.3, 2.4, 2.5, 3.1 to 3.3 and Part 4 are in `rag_analysis.md`, built from the tables
above and the CSV files in `results/`.

In [22]:
run_info = dict(dry_run=DRY_RUN, device=DEVICE, platform=platform.platform(), generator=GEN_MODEL,
                embedding_models=EMBEDDING_MODELS, top_k=TOP_K, max_new_tokens=MAX_NEW_TOKENS,
                best_config=list(best), weakest_config=list(worst))
(RESULTS_DIR / 'run_info.json').write_text(json.dumps(run_info, indent=1))
print(json.dumps(run_info, indent=1))

{
 "dry_run": false,
 "device": "mps",
 "platform": "macOS-26.6.2-arm64-arm-64bit-Mach-O",
 "generator": "HuggingFaceTB/SmolLM2-1.7B-Instruct",
 "embedding_models": {
  "small": "sentence-transformers/all-MiniLM-L6-v2",
  "medium": "sentence-transformers/all-mpnet-base-v2",
  "large": "BAAI/bge-large-en-v1.5"
 },
 "top_k": 3,
 "max_new_tokens": 120,
 "best_config": [
  "large",
  "fixed"
 ],
 "weakest_config": [
  "small",
  "hybrid"
 ]
}
